# 05. 벡터 스토어 (Vector Store)

임베딩 벡터를 저장하고, 쿼리와 **가장 유사한 문서를 빠르게 검색**합니다.

## FAISS란?
Facebook AI Research에서 만든 벡터 유사도 검색 라이브러리입니다.  
로컬에서 빠르게 동작하며 실습에 적합합니다.

In [ ]:
#  uv add faiss-cpu

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 준비
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"청크 수: {len(chunks)}")

청크 수: 10


## 1. FAISS 벡터 스토어 생성

`from_documents()`는 청크를 임베딩하고 벡터 스토어에 저장합니다.

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
    # distance_strategy=DistanceStrategy.EUCLIDEAN # <-- 유클리드 거리 기준 (기본값)
    # distance_strategy=DistanceStrategy.COSINE,
)

print("벡터 스토어 생성 완료")
print(f"저장된 벡터 수: {vector_store.index.ntotal}")




벡터 스토어 생성 완료
저장된 벡터 수: 10


## 2. 유사도 검색 - similarity_search()

In [5]:
query = "RAG란 무엇인가요?"
results = vector_store.similarity_search(query, k=3)

results
# print(f"검색 결과 {len(results)}개\n")
# for i, doc in enumerate(results):
#     print(f"[{i+1}] {doc.page_content[:150]}")
#     print(f"     출처: {doc.metadata}")
#     print()




[Document(id='de59601a-5518-4533-9808-0942dc261562', metadata={'source': 'data/ai_basic.txt'}, page_content='RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?'),
 Document(id='c505e9a0-b2f2-43d7-9e0b-98173ccec8d6', metadata={'source': 'data/ai_basic.txt'}, page_content='대규모 언어 모델(LLM)이란?\n대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언어 작업을 수행할 수 있습니다.\nLLM의 한계로는 학습 데이터 이후의 최신 정보를 모른다는 점, 할루시네이션(Hallucination) 현상으로 사실과 다른 내용을 생성할 수 있다는 점이 있습니다.\nRAG(Retrieval-Augmented Generation)이란?'),
 Document(id='76b2a98c-41ef-4709-b852-54baec1ace55', metadata={'source': 'data/ai_basic.txt'}, page_content='LangChain이란?\nLangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트

## 3. 유사도 점수 포함 검색 - similarity_search_with_score()

In [7]:
results_with_score = vector_store.similarity_search_with_score(query, k=3)

results_with_score

# for doc, score in results_with_score:
#     print(f"거리 점수: {score:.4f}")
#     print(f"내용: {doc.page_content[:100]}")
#     print()




[(Document(id='de59601a-5518-4533-9808-0942dc261562', metadata={'source': 'data/ai_basic.txt'}, page_content='RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?'),
  np.float32(0.9450097)),
 (Document(id='c505e9a0-b2f2-43d7-9e0b-98173ccec8d6', metadata={'source': 'data/ai_basic.txt'}, page_content='대규모 언어 모델(LLM)이란?\n대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언어 작업을 수행할 수 있습니다.\nLLM의 한계로는 학습 데이터 이후의 최신 정보를 모른다는 점, 할루시네이션(Hallucination) 현상으로 사실과 다른 내용을 생성할 수 있다는 점이 있습니다.\nRAG(Retrieval-Augmented Generation)이란?'),
  np.float32(1.3141618)),
 (Document(id='76b2a98c-41ef-4709-b852-54baec1ace55', metadata={'source': 'data/ai_basic.txt'}, page_content='LangChain이란?\nLangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등

> FAISS의 점수는 **L2 거리(낮을수록 유사)**입니다.

## 4. 벡터 스토어 저장 및 불러오기

In [8]:
# 로컬에 저장

vector_store.save_local("data/faiss_index")
print("저장 완료")

저장 완료


In [9]:
# 저장된 인덱스 불러오기
loaded_store = FAISS.load_local(
    "data/faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # 보안 위험 경고 무시 (로컬에서만 사용)
)

# 검색 테스트
results = loaded_store.similarity_search("임베딩이 뭔가요?", k=2)
for doc in results:
    print(doc.page_content[:150])
    print()

임베딩(Embedding)이란?
임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.


RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.
RAG 파이프라인은 문서 로딩 



## 5. 문서 추가

In [10]:
from langchain_core.documents import Document

new_docs = [
    Document(
        page_content="LangSmith는 LLM 애플리케이션의 디버깅과 모니터링 도구입니다.",
        metadata={"source": "manual"}
    )
]

vector_store.add_documents(new_docs)
print(f"추가 후 벡터 수: {vector_store.index.ntotal}")


추가 후 벡터 수: 11


## 정리

| 메서드 | 설명 |
|---|---|
| `from_documents()` | 문서 임베딩 후 저장 |
| `similarity_search()` | 유사 문서 검색 |
| `similarity_search_with_score()` | 유사도 점수 포함 검색 |
| `save_local()` / `load_local()` | 저장/불러오기 |
| `add_documents()` | 문서 추가 |
